we can store:

```
(vpn1, ppn1)
(vpn2, ppn2)
(vpn3, ppn3)
```

but when CPU tries to load a vpn_x, it has to loop through the entire page table, which is slow.

So we can use vpn number as index.

If vpn bits are 27 bits and 12 bits for offset (since 1 page 4kb), then we need
>2^27 entries!

times size per entries, ram needed is too much just for page table.

instead, we use 3 levels, 9 + 9 + 9 bits, lookup is more work. But each level 9 bits is 2^9 = 512 entries, each page table entry (PTE) can have 8 bytes, so we can use 1 page to store a page table. Initially, storing 1 entry needs to setup 3 page table, so 4k * 3 = 12k ram space. But it's a lot more efficient than 1 level.

$$
VPN=L_2\times512^2+L_1\times512+L_0
$$

---


However, **Physical Address Space ≠ RAM**.

The key is to distinguish between two things:

- **RAM**: The actual hardware that stores data.
- **Physical Address Space**: A set of addresses that the CPU uses to access RAM and other hardware devices.

## Understanding from First Principles

Suppose the CPU issues a request:

```text
CPU: I want to read physical address 0x80001000
```

The address itself does not tell the CPU whether it refers to RAM or another device.

The hardware system determines where to route the request based on the address.

```text
                     CPU
                      |
                Physical Address
                      |
                Hardware Interconnect
                      |
          +-----------+-----------+
          |                       |
     0x80000000              0x10000000
          |                       |
         RAM                     UART
    (Stores Data)          (Serial Device)
```

For example, in the QEMU RISC-V `virt` machine used by xv6:

| Physical Address | Corresponding Hardware |
|---|---|
| `0x10000000` | UART |
| `0x10001000` | VirtIO Disk Controller |
| `0x80000000` | Start of RAM |
| `0x80001000` | Inside RAM |
| `0x88000000` | Default upper RAM limit in xv6 (exclusive) |

Therefore, **accessing a physical address does not necessarily mean accessing RAM**.

---

You've identified two concepts that are easy to confuse:

1. **Each address corresponds to 1 byte of RAM.**
2. **A pointer in xv6 (64-bit RISC-V) occupies 8 bytes = 64 bits.**

These two facts do not contradict each other.

Suppose RAM starts at address 0:

```text
RAM
Address     Data
0           1 byte
1           1 byte
2           1 byte
...
233         1 byte
234         1 byte
235         1 byte
...
```

The byte at index 234 (assuming zero-based indexing) has the address `234`.

If we store this address in a 64-bit pointer:

```text
Address = 234

Binary (64 bits):
00000000 00000000 00000000 00000000
00000000 00000000 00000000 11101010
```

Although the number 234 requires only 8 bits to represent, **a pointer in xv6 always uses 64 bits to store the address**.

---

Page is how OS orangize RAM, not how OS reads from RAM.

**Allocating a page does not mean the CPU reads 4 KiB from RAM.** It simply updates the memory management metadata. In addition, xv6's `kalloc()` uses `memset()` to initialize the entire page.

---

1. **RAM is a sequence of bytes, and each byte has an address.**
2. **A pointer is a variable that stores an address.**
3. **The pointer type tells the compiler how to interpret the data at that address.**

Once you understand these three concepts, `int **pp` is no longer mysterious.

For example, pointers can be taught in this way:

1. RAM is a sequence of bytes, and each byte has an address.
2. I want to keep track of where a variable is stored, so we introduce **pointers**.
3. I want to access the variable through its address, so we introduce the **dereference operator `*`**.
4. I want to know where the pointer itself is stored, so we introduce **`&p`**.
5. A pointer is itself a variable, so naturally, we can have a **pointer to a pointer**.
6. But an address is just a number. The compiler needs to know how many bytes to read and how to interpret them, so we introduce **pointer types**.

By the sixth step, students can even derive for themselves why something like `int **` is necessary.


